# Mosaic AI Agent Framework: Author and deploy a multi-agent system with Genie

This notebook demonstrates how to build a multi-agent system using Mosaic AI Agent Framework and [LangGraph](https://blog.langchain.dev/langgraph-multi-agent-workflows/), where [Genie](https://www.databricks.com/product/ai-bi/genie) is one of the agents.
In this notebook, you:
1. Author a multi-agent system using LangGraph.
1. Wrap the LangGraph agent with MLflow `ChatAgent` to ensure compatibility with Databricks features.
1. Manually test the multi-agent system's output.
1. Log and deploy the multi-agent system.

This example is based on [LangGraph documentation - Multi-agent supervisor example](https://github.com/langchain-ai/langgraph/blob/main/docs/docs/tutorials/multi_agent/agent_supervisor.ipynb)

## Why use a Genie agent?

Multi-agent systems consist of multiple AI agents working together, each with specialized capabilities. As one of those agents, Genie allows users to interact with their structured data using natural language.

Unlike SQL functions which can only run pre-defined queries, Genie has the flexibility to create novel queries to answer user questions.

## Prerequisites

- Address all `TODO`s in this notebook.
- Create a Genie Space, see Databricks documentation ([AWS](https://docs.databricks.com/aws/genie/set-up) | [Azure](https://learn.microsoft.com/azure/databricks/genie/set-up)).

In [0]:
# %pip install -U -qqq mlflow langgraph==0.3.4 databricks-langchain==0.5.1 databricks-agents==0.22.1 uv
# dbutils.library.restartPython()

In [0]:
%pip install -U -qqq mlflow-skinny[databricks] langgraph==0.3.4 databricks-langchain databricks-agents uv
dbutils.library.restartPython()
# DBR 16.4 LTS


## Define the multi-agent system

Create a multi-agent system in LangGraph using a supervisor agent node directing the following agent nodes:
- **GenieAgent**: The Genie agent that queries and reasons over structured data.
- **Tool-calling agent**: An agent that calls Unity Catalog function tools.

In this example, the tool-calling agent uses the built-in Unity Catalog function `system.ai.python_exec` to execute Python code.
For examples of other tools you can add to your agents, see Databricks documentation ([AWS](https://docs.databricks.com/aws/generative-ai/agent-framework/agent-tool) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/agent-tool)).


#### Wrap the LangGraph agent using the `ChatAgent` interface

Databricks recommends using `ChatAgent` to ensure compatibility with Databricks AI features and to simplify authoring multi-turn conversational agents using an open source standard. 

The `LangGraphChatAgent` class implements the `ChatAgent` interface to wrap the LangGraph agent.

See MLflow's [ChatAgent documentation](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ChatAgent).

#### Write agent code to file

Define the agent code in a single cell below. This lets you write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.


In [0]:
%%writefile agent.py
import functools
import os
from typing import Any, Generator, Literal, Optional

import mlflow
from databricks.sdk import WorkspaceClient
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    VectorSearchRetrieverTool
)
from databricks_langchain.genie import GenieAgent
from langchain_core.runnables import RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import create_react_agent
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentChunk,
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)
from pydantic import BaseModel
from databricks.sdk.credentials_provider import ModelServingUserCredentials


from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


###################################################
## Create a RAG Agent
###################################################

# TODO fill in fields below

faq_index = {
    "index_name": "vr_demo.playground.faq_index",
    "description": "Use esta função para responder perguntas sobre prazos de entrega, pedidos de troca ou devolução, entre outras perguntas frequentes sobre o nosso marketplace.",
    "name": "faq_index",
    "columns": ["resposta"],
    "num_results": 1,
    "host": "https://e2-demo-field-eng.cloud.databricks.com"
}

def create_vs_tool(index_name, description, name, columns, num_results, host):
    # Use user authenticated client to initialize a vector search retrieval tool
    user_authenticated_client = WorkspaceClient(
        host=host,
        credentials_strategy=ModelServingUserCredentials()
    )
    vs_tool = VectorSearchRetrieverTool(
        index_name=index_name,
        description=description,
        tool_name=name,
        columns=columns,
        num_results=num_results,
        workspace_client=user_authenticated_client
    )
    return vs_tool

def create_rag_agent(index_name, description, name, columns, num_results, host):
    faq_tool = create_vs_tool(index_name, description, name, columns, num_results, host)

    # # Select our Databricks Foundation Model
    # llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0.1)

    # # Define our prompt
    # template = '''
    # Você é um assistente especialista e pode responder perguntas sobre prazos de entrega, pedidos de troca ou devolução, entre outras sobre o nosso marketplace.
    # Se a pergunta não for sobre um desses assuntos, educadamente responda que você não pode responder a este tipo de pergunta.
    # Use o contexto abaixo para responder às perguntas. Se o contexto não fornecer uma resposta satisfatória, apenas diga que você não sabe. Não tente inventar uma resposta.

    # Contexto: {context}

    # Pergunta: {question}

    # Resposta: '''

    # # Create a prompt template
    # prompt = PromptTemplate(
    #     template=template, 
    #     input_variables=[
    #         'context', 
    #         'question',
    #     ]
    # )

    # Define a function to join the retrieve documents
    def format_docs(docs):
        return "\n\n".join(doc.metadata['resposta'] for doc in docs)

    # Chain all steps together
    # return (
    #     {
    #         "context": faq_tool | format_docs,
    #         "question": RunnablePassthrough(),
    #     }
    #     | prompt
    #     | llm
    #     | StrOutputParser()
    # )

    return (
        faq_tool
        | format_docs
        | StrOutputParser()
    )

def rag_node(state, agent, name):
    result = agent.invoke(state["messages"][-1]["content"])
    return {
        "messages": [
            {
                "role": "assistant",
                "content": f"Contexto: {result}",
                "name": name,
            }
        ]
    }


###################################################
## Create a GenieAgent with access to a Genie Space
###################################################

# TODO add GENIE_SPACE_ID and a description for this space
# You can find the ID in the URL of the genie room /genie/rooms/<GENIE_SPACE_ID>

sales_genie = {
    "genie_space_id": "01f058479ad31fd0b21551c3ce350db9",
    "name": "SalesGenie",
    "description": "Use esta ferramenta para responder perguntas sobre vendas",
    "host": "https://e2-demo-field-eng.cloud.databricks.com"
}

log_genie = {
    "genie_space_id": "01f06eccbb7f1f6bae899688b4a95376",
    "name": "LogGenie",
    "description": "Use esta ferramenta para responder perguntas sobre logística e estoque",
    "host": "https://e2-demo-field-eng.cloud.databricks.com"
}

def create_genie_agent(genie_space_id, name, description, host):
    # Use user authenticated client to initialize a vector search retrieval tool
    user_authenticated_client = WorkspaceClient(
        host=host,
        credentials_strategy=ModelServingUserCredentials()
    )

    return GenieAgent(
        genie_space_id=genie_space_id,
        genie_agent_name=name,
        description=description,
        client=user_authenticated_client
    )


############################################
# Define your LLM endpoint and system prompt
############################################

# TODO: Replace with your model serving endpoint
# multi-agent Genie works best with claude 3.7 or gpt 4o models.
LLM_ENDPOINT_NAME = "databricks-claude-3-7-sonnet"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)


#############################
# Define the supervisor agent
#############################

# TODO update the max number of iterations between supervisor and worker nodes
# before returning to the user
MAX_ITERATIONS = 3

worker_descriptions = {
    faq_index["name"]: faq_index["description"],
    sales_genie["name"]: sales_genie["description"],
    log_genie["name"]: log_genie["description"]
}

formatted_descriptions = "\n".join(
    f"- {name}: {desc}" for name, desc in worker_descriptions.items()
)

system_prompt = f"Escolha entre encaminhar para um dos trabalhadores abaixo ou encerrar a conversa se uma resposta for fornecida. \n{formatted_descriptions}"
options = ["FINISH"] + list(worker_descriptions.keys())
FINISH = {"next_node": "FINISH"}

def supervisor_agent(state):
    count = state.get("iteration_count", 0) + 1
    if count > MAX_ITERATIONS:
        return FINISH
    
    class nextNode(BaseModel):
        next_node: Literal[tuple(options)]

    preprocessor = RunnableLambda(
        lambda state: [{"role": "system", "content": system_prompt}] + state["messages"]
    )
    supervisor_chain = preprocessor | llm.with_structured_output(nextNode)
    next_node = supervisor_chain.invoke(state).next_node
    
    # if routed back to the same node, exit the loop
    if state.get("next_node") == next_node:
        return FINISH
    return {
        "iteration_count": count,
        "next_node": next_node
    }

#######################################
# Define our multiagent graph structure
#######################################


def agent_node(state, agent, name):
    result = agent.invoke(state)
    return {
        "messages": [
            {
                "role": "assistant",
                "content": result["messages"][-1].content,
                "name": name,
            }
        ]
    }


def final_answer(state):
    prompt = "Usando apenas o conteúdo das mensagens anteriores como contexto, responda à pergunta do usuário. Não adicione nenhum comentário sobre já ter enviado a resposta anteriormente."
    preprocessor = RunnableLambda(
        lambda state: state["messages"] + [{"role": "user", "content": prompt}]
    )
    final_answer_chain = preprocessor | llm
    return {"messages": [final_answer_chain.invoke(state)]}


class AgentState(ChatAgentState):
    next_node: str
    iteration_count: int


def create_agent():
    # tools = [create_vs_tool(**faq_index)]
    # faq_index_node = ChatAgentToolNode(tools)

    # nodes = []
    faq_index_node = functools.partial(rag_node, agent=create_rag_agent(**faq_index), name=faq_index["name"])
    sales_genie_node = functools.partial(agent_node, agent=create_genie_agent(**sales_genie), name=sales_genie["name"])
    log_genie_node = functools.partial(agent_node, agent=create_genie_agent(**log_genie), name=log_genie["name"])
    # node.append(functools.partial(rag_node, agent=create_rag_agent(**faq_index), name=faq_index["name"]))
    # node.append(functools.partial(agent_node, agent=create_genie_agent(**sales_genie), name=sales_genie["name"]))
    # node.append(functools.partial(agent_node, agent=create_genie_agent(**cust_genie), name=cust_genie["name"]))

    workflow = StateGraph(AgentState)
    workflow.add_node("supervisor", supervisor_agent)
    workflow.add_node(faq_index["name"], faq_index_node)
    workflow.add_node(sales_genie["name"], sales_genie_node)
    workflow.add_node(log_genie["name"], log_genie_node)
    # for node in nodes:
    #     workflow.add_node(faq_index["name"], faq_index_node)
    workflow.add_node("final_answer", final_answer)

    workflow.set_entry_point("supervisor")
    # We want our workers to ALWAYS "report back" to the supervisor when done
    for worker in worker_descriptions.keys():
        workflow.add_edge(worker, "supervisor")

    # Let the supervisor decide which next node to go
    workflow.add_conditional_edges(
        "supervisor",
        lambda x: x["next_node"],
        {**{k: k for k in worker_descriptions.keys()}, "FINISH": "final_answer"},
    )
    workflow.add_edge("final_answer", END)
    return workflow.compile()

###################################
# Wrap our multi-agent in ChatAgent
###################################


class LangGraphChatAgent(ChatAgent):
    # def __init__(self, agent: CompiledStateGraph):
    #     self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        
        agent = create_agent()

        request = {
            "messages": [m.model_dump_compat(exclude_none=True) for m in messages]
        }

        messages = []
        for event in agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                messages.extend(
                    ChatAgentMessage(**msg) for msg in node_data.get("messages", [])
                )
        return ChatAgentResponse(messages=messages)

    def predict_stream(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> Generator[ChatAgentChunk, None, None]:
        
        agent = create_agent()

        request = {
            "messages": [m.model_dump_compat(exclude_none=True) for m in messages]
        }
        for event in agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (
                    ChatAgentChunk(**{"delta": msg})
                    for msg in node_data.get("messages", [])
                )


# Create the agent object, and specify it as the agent object to use when
# loading the agent back for inference via mlflow.models.set_model()
mlflow.langchain.autolog()
AGENT = LangGraphChatAgent()
mlflow.models.set_model(AGENT)

## Test the agent

Interact with the agent to test its output. Since this notebook called `mlflow.langchain.autolog()` you can view the trace for each step the agent takes.

**TODO**: Replace this placeholder `input_example` with a domain-specific prompt for your agent.

In [0]:
dbutils.library.restartPython()

In [0]:
from agent import AGENT

In [0]:
AGENT.predict({
    "messages": [
        {
            "role": "user",
            "content": "Como solicitar uma troca?",
        }
    ]
})

In [0]:
input_example = {
    "messages": [
        {
            "role": "user",
            "content": "Qual a quantidade de itens vendidos em out/22?",
        }
    ]
}
AGENT.predict(input_example)

In [0]:
AGENT.predict({
    "messages": [
        {
            "role": "user",
            "content": "Qual o estoque do PRODUCT 6257 em 01-01-2021?",
        }
    ]
})

In [0]:
for event in AGENT.predict_stream(input_example):
  print(event, "-----------\n")

## Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model().`
  - **TODO**: If your Unity Catalog tool queries a [vector search index](docs link) or leverages [external functions](docs link), you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources)).

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from agent import LLM_ENDPOINT_NAME
from databricks_langchain import UnityCatalogTool, VectorSearchRetrieverTool
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksGenieSpace,
    DatabricksServingEndpoint,
)
from pkg_resources import get_distribution
from mlflow.models.auth_policy import AuthPolicy, SystemAuthPolicy, UserAuthPolicy

# TODO: Manually include underlying resources if needed. See the TODO in the markdown above for more information.
resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)
]
systemAuthPolicy = SystemAuthPolicy(resources=resources)

# TODO: Manually include the required api scopes for this authorization.
userAuthPolicy = UserAuthPolicy(api_scopes=[
        "serving.serving-endpoints", 
        "dashboards.genie",
        "vectorsearch.vector-search-endpoints",
        "vectorsearch.vector-search-indexes"
    ]
)

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model="agent.py",
        input_example=input_example,
        auth_policy=AuthPolicy(system_auth_policy=systemAuthPolicy, user_auth_policy=userAuthPolicy),
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
        ]
    )

## Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks))."

In [0]:
# mlflow.models.predict(
#     model_uri=f"runs:/{logged_agent_info.run_id}/agent",
#     input_data=input_example,
#     env_manager="uv",
# )

## Register the model to Unity Catalog

Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "vr_demo"
schema = "playground"
model_name = "vr_agent_genie_vs_obo"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

## Deploy the agent

In [0]:
# # ONLY FOR GROUP CLUSTERS
# import os

# os.environ['DATABRICKS_HOST'] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
# os.environ['DATABRICKS_TOKEN'] = dbutils.secrets.get('my_secret_scope', 'vr-genie-secret')

In [0]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    endpoint_name=model_name+'_2',
    tags={'RemoveAfter':'2026-01-01'},
    budget_policy_id='a6633f5f-76b2-4db8-959b-90f128137243'
)

## Next steps

After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/deploy-agent.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/deploy-agent)).

In [0]:
# %pip install -U -qqq mlflow langgraph==0.3.4 databricks-langchain==0.5.1 databricks-agents==0.22.1 uv
# dbutils.library.restartPython()

In [0]:
# catalog = "vr_demo"
# schema = "playground"
# model_name = "vr_agent_genie_vs_obo"
# UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"
# model_version = 1

# from databricks import agents

# agents.deploy(
#     UC_MODEL_NAME,
#     model_version,
#     endpoint_name=model_name,
# )

In [0]:
# %pip install -U -qqq mlflow langgraph==0.3.4 databricks-langchain databricks-agents uv
# dbutils.library.restartPython()

In [0]:
# from databricks import agents

# catalog = "vr_demo"
# schema = "playground"
# model_name = "vr_agent_genie_obo"
# UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# agents.set_permissions(model_name=UC_MODEL_NAME, users=["erico.silva@databricks.com"], permission_level=agents.PermissionLevel.CAN_QUERY)